In [ ]:
import pandas as pd
from scipy.stats import mannwhitneyu, f_oneway
from pandas.api.types import is_numeric_dtype


def get_features_cat_regression(
    df: pd.DataFrame,
    target_col: str,
    pvalue: float = 0.05
) -> list:

    # Comprobaciones de entrada
    if not isinstance(df, pd.DataFrame):
        print("Error: df debe ser un pd.DataFrame")
        return None

    if target_col not in df.columns:
        print(f"Error: '{target_col}' no existe en el DataFrame")
        return None

    if not is_numeric_dtype(df[target_col]):
        print(f"Error: '{target_col}' debe ser numérica")
        return None

    if not isinstance(pvalue, float) or not (0 < pvalue < 1):
        print("Error: pvalue debe ser un float entre 0 y 1")
        return None

    columnas_significativas = []

    # Seleccionar columnas categóricas
    for col in df.columns:

        if col == target_col:
            continue

        temp_df = df[[col, target_col]].dropna()

        n_categorias = temp_df[col].nunique()

        cardinalidad_relativa = n_categorias / len(temp_df)

        if n_categorias < 20 and cardinalidad_relativa < 0.2:
        # Consideramos categórica si tiene menos de 20 categorías 
        # o una cartdinalidad relativa menor que 0.2

            categorias = temp_df[col].unique()

        # EXACTAMENTE 2 categorías -> Mann-Whitney U
        if len(categorias) == 2:

            grupo1 = temp_df[temp_df[col] == categorias[0]][target_col]
            grupo2 = temp_df[temp_df[col] == categorias[1]][target_col]

            _, p_valor = mannwhitneyu(grupo1, grupo2)

            if p_valor < pvalue:
                columnas_significativas.append(col)

        # MÁS DE 2 categorías -> ANOVA
        elif len(categorias) > 2:

            grupos = [
                temp_df[temp_df[col] == categoria][target_col]
                for categoria in categorias
            ]

            _, p_valor = f_oneway(*grupos)

            if p_valor < pvalue:
                columnas_significativas.append(col)

    return columnas_significativas